In [ ]:
class TaxiDataExtractor:
    

In [11]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("TaxiETL").getOrCreate()

bronze = spark.read.parquet("data/raw/yellow_tripdata_2024-01.parquet")
bronze.printSchema()
bronze.show(5)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------

In [20]:
bronze.select('tpep_pickup_datetime').show()

+--------------------+
|tpep_pickup_datetime|
+--------------------+
| 2024-01-01 00:57:55|
| 2024-01-01 00:03:00|
| 2024-01-01 00:17:06|
| 2024-01-01 00:36:38|
| 2024-01-01 00:46:51|
| 2024-01-01 00:54:08|
| 2024-01-01 00:49:44|
| 2024-01-01 00:30:40|
| 2024-01-01 00:26:01|
| 2024-01-01 00:28:08|
| 2024-01-01 00:35:22|
| 2024-01-01 00:25:00|
| 2024-01-01 00:35:16|
| 2024-01-01 00:43:27|
| 2024-01-01 00:51:53|
| 2024-01-01 00:50:09|
| 2024-01-01 00:41:06|
| 2024-01-01 00:52:09|
| 2024-01-01 00:56:38|
| 2024-01-01 00:32:34|
+--------------------+
only showing top 20 rows



In [32]:
from pyspark.sql import functions as F

bronze_z_czasem = bronze.withColumn(
    "trip_duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
)


oznaczone = bronze_z_czasem.withColumn(
    "reason",
    F.when(bronze_z_czasem['trip_duration_min'] < 3, "too_short")
     .when(bronze_z_czasem['trip_duration_min'] > 120, "too_long")
     .when(bronze_z_czasem['trip_distance'] <= 0, "bad_distance")
     .when(bronze_z_czasem['fare_amount'] <= 0, "bad_price")
     .otherwise(None)
)

oznaczone.groupBy("reason").count().show()

+------------+-------+
|      reason|  count|
+------------+-------+
|   too_short| 121249|
|   bad_price|  29666|
|        NULL|2777697|
|    too_long|   2742|
|bad_distance|  33270|
+------------+-------+



In [29]:
print("Bronze:", bronze.count())
print("Silver (po filtrach, przed percentylem):", silver.count())

Bronze: 2964624
Silver (po filtrach, przed percentylem): 2777697
